# R08 · Heterogeneous SoCs - spinlocks across cores, remoteproc, RPMsg, shared-memory rings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/05_realtime_linux/R08_heterogeneous_socs_spinlocks_across_cores_remote.ipynb)

**Track:** Real-time and Linux · **Skill axis (sheet):** Real-time and Linux · **Time:** about 65 min

**Skill Hub concepts:** Memory ordering and spinlocks across cores; Asymmetric multiprocessing; Heterogeneous multicore system on chip; Remote processor framework and boot of a microcontroller core; RPMsg and OpenAMP; Shared-memory ring buffer with doorbell

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("R08")

## Map: Linux and an RTOS on one chip (STM32MP1, i.MX 8M, AM62, Zynq)

```
  Cortex-A cores (SMP Linux)                         Cortex-M core (FreeRTOS / Zephyr / bare metal)
  ROS 2, vision, networking                          1-10 kHz motor control, hard real time
       │  remoteproc: load firmware ELF, parse its resource table, release the core from reset
       │  echo rdl_motor.elf > /sys/class/remoteproc/remoteproc0/firmware ; echo start > .../state
       └──── RPMsg over virtio: two vrings (rings of buffer descriptors) in shared DDR + a mailbox interrupt ────┘
             Linux side: /dev/rpmsg0 or /dev/ttyRPMSG0          M side: OpenAMP library (rpmsg_send / callback)
```

| Term | Meaning |
|---|---|
| SMP | one OS across identical cores |
| AMP | separate OS images per core (or core cluster), each owning its peripherals |
| Heterogeneous SoC | different core types on one die: big application cores + microcontroller cores + DSP/NPU |
| Doorbell | a mailbox/IPI interrupt that says "look at the shared ring", so the receiver need not poll |

**Spinlocks across cores** need atomic read-modify-write plus ordering: take the lock with an **acquire**, release it
with a **release**, so the protected data is visible to the next owner. A **ticket lock** (take a number, wait for it to
be served) is fair: no core starves.

```
 ticket lock:   me = fetch_add(&next, 1)        while (load_acquire(&serving) != me) relax();
                ... critical section ...        store_release(&serving, serving + 1);
```

## Task 1: a ticket spinlock in C11

In [ ]:
%%writefile ticket.c
#include <stdatomic.h>
#include <sched.h>
typedef struct { atomic_uint next; atomic_uint serving; } ticket_lock_t;

void tl_lock(ticket_lock_t *l) {
    (void)l; /* TODO */
}

void tl_unlock(ticket_lock_t *l) {
    (void)l; /* TODO */
}

In [ ]:
lab.check("ticket")

## Task 2: a shared-memory ring with a doorbell, between two processes
This models the Linux and Cortex-M sides: a header with `head` and `tail` (uint32 at byte offsets 0 and 4) and
`N` slots of 64 bytes (byte 0 = length, then data) in shared memory, plus an `eventfd` as the doorbell interrupt.
`send` waits while the ring is full, writes the slot, **then** advances `head`, then rings the doorbell.
`recv_all` takes everything between `tail` and `head`, then advances `tail`.

In [ ]:
import os, struct
from multiprocessing import shared_memory

N, SLOT = 16, 64

class Ring:
    def __init__(self, name=None):
        self.shm = shared_memory.SharedMemory(name=name, create=name is None, size=8 + N * SLOT)
        if name is None:
            self.shm.buf[:8] = bytes(8)
        self.doorbell = None
    def head(self): return struct.unpack_from("<I", self.shm.buf, 0)[0]
    def tail(self): return struct.unpack_from("<I", self.shm.buf, 4)[0]
    def set_head(self, v): struct.pack_into("<I", self.shm.buf, 0, v & 0xFFFFFFFF)
    def set_tail(self, v): struct.pack_into("<I", self.shm.buf, 4, v & 0xFFFFFFFF)
    def slot(self, i): return 8 + (i % N) * SLOT

def send(ring, data: bytes):
    raise NotImplementedError  # TODO

def recv_all(ring):
    raise NotImplementedError  # TODO

lab.check("ring", Ring, send, recv_all)

In [ ]:
%%writefile r08_ring.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/r08.h
// Grade it with lab.check("ring", "r08_ring.c"). For C++, name the file .cpp in both places.
#include "r08.h"

void ring_send(uint8_t *shm, int doorbell, const uint8_t *data, size_t len) {
    (void)shm; (void)doorbell; (void)data; (void)len; /* TODO: wait while full, copy, publish head (release), eventfd */
}

size_t ring_recv_all(uint8_t *shm, uint8_t *out, size_t out_cap, size_t *lens, size_t max_msgs) {
    (void)shm; (void)out; (void)out_cap; (void)lens; (void)max_msgs;
    return 0; /* TODO: copy slots from tail to head, then publish tail (release) */
}

In [ ]:
# lab.check("ring", "r08_ring.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** An RPMsg vring with 256 buffers of 512 bytes. Bytes of shared memory for buffers?

**M2 (Apply).** Ticket lock: you drew ticket 7, `serving` is 4. How many unlocks must happen before you enter?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Evaluate).** Why run the 10 kHz current loop on the Cortex-M core of an STM32MP1 instead of a Linux thread?
- a) Linux cannot do floating point
- b) the M core has deterministic interrupt latency and no OS jitter; Linux handles networking, vision and ROS 2, and the two talk over RPMsg
- c) the A cores cannot access the timers

**K2 (Understand).** Walk through how Linux boots the Cortex-M firmware with remoteproc and how the first RPMsg channel
appears in user space.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `r08.h`).
3. In `roboprep/checks/r08.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()